## Retriever and chain with Langchain

In [49]:
# Pdf reader (also part of loading data phase)
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('sample.pdf')
docs=loader.load()
docs

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-05-19T10:30:11-07:00', 'moddate': '2020-05-19T10:42:37-07:00', 'source': 'sample.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Get star t ed with\nMicr oso f t OneDriv e\nSave your files to OneDrive to keep them protected, \nbacked up, and accessible from all your devices, anywhere.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-05-19T10:30:11-07:00', 'moddate': '2020-05-19T10:42:37-07:00', 'source': 'sample.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}, page_content='Anywher e access\nOneDrive.com\nOneDrive mobile app\nWith  and the\nyou can\ncreate, access and edit your files \nfrom all your devices, virtually \nanywhere you happen to be.\nCloud st orage\nLearn how to upload files\nOneDrive provides one secured \nplace for your files and photos. \u2028\nS tart with 5 GB of free storage or \nupgrade to Microsoft 365 for 1 TB.\nPC fold

In [50]:
# step2 : Splitting the documents into smaller chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunk_documents=text_splitter.split_documents(docs)
chunk_documents


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-05-19T10:30:11-07:00', 'moddate': '2020-05-19T10:42:37-07:00', 'source': 'sample.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='Get star t ed with\nMicr oso f t OneDriv e\nSave your files to OneDrive to keep them protected, \nbacked up, and accessible from all your devices, anywhere.'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2020-05-19T10:30:11-07:00', 'moddate': '2020-05-19T10:42:37-07:00', 'source': 'sample.pdf', 'total_pages': 5, 'page': 1, 'page_label': '2'}, page_content='Anywher e access\nOneDrive.com\nOneDrive mobile app\nWith  and the\nyou can\ncreate, access and edit your files \nfrom all your devices, virtually \nanywhere you happen to be.\nCloud st orage\nLearn how to upload files\nOneDrive provides one secured \nplace for your files and photos. \u2028\nS tart with 5 GB of free storage or \nupgrade to Microsoft 365 for 1 TB.\nPC fold

In [51]:
# step 3 : vector Embedding and Vector store( converting words into numerical vectors called embeddings and storing then in a vector store)
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

db= FAISS.from_documents(chunk_documents, OpenAIEmbeddings())
db
retriever = db.as_retriever()

In [52]:
#step4 :  in this step basically we are doing the retrieval part where we are giving the query and based on the similarity of the query with the documents in the vector store it will return the most relevant document to us
query = "Share documents, folders, anf photos with anyone"
retrieved_result = db.similarity_search(query)
print(retrieved_result[0].page_content)


Anywher e access
OneDrive.com
OneDrive mobile app
With  and the
you can
create, access and edit your files 
from all your devices, virtually 
anywhere you happen to be.
Cloud st orage
Learn how to upload files
OneDrive provides one secured 
place for your files and photos.  
S tart with 5 GB of free storage or 
upgrade to Microsoft 365 for 1 TB.
PC folder b ackup
How to set up PC folder backup
Turn on PC folder backup to 
automatically back up and sync  
your Desktop, Documents, and 
Pictures folders to OneDrive.


In [53]:
## Design chatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the following question based only on the provided context.
Think step by step before providing an answer.
<context>
{context}
</context>"""),
    ("human", "{input}")
])


In [54]:

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo")

In [55]:
retirever = db.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000016CDDF520F0>, search_kwargs={})

In [56]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# 1. Create the document chain
# Note: Renamed to 'document_chain' to avoid conflicting with the final 'chain' variable
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# 2. Create the retrieval chain
# Note: Change 'combine_documents_chain' to 'combine_docs_chain' to match the classic library requirements
retrieval_chain = create_retrieval_chain(retriever=retriever, combine_docs_chain=document_chain)

retrieval_chain

response = retrieval_chain.invoke({"input": "how can i share documents with anyone?"})
response['answer']




'To share documents with anyone using OneDrive, you can follow these steps:\n\n1. Access OneDrive either through OneDrive.com or the OneDrive mobile app.\n2. Locate the document you want to share.\n3. Right-click on the document or select the document and look for the sharing option.\n4. Choose the option to share the document.\n5. Enter the email address of the person you want to share the document with.\n6. Set the permissions (view only or edit) for the recipient.\n7. Send the share link to the recipient.\n8. The recipient can then click on the link to view or edit the document, even without having an account.\n\nBy following these steps, you can easily share documents with anyone using OneDrive.'